# Explore GSE123813 BCC T-cell Dataset (Yost et al. 2019)

**Dataset**: Yost et al. 2019 — T cells from BCC (basal cell carcinoma) patients,  
single-cell RNA-seq + TCR-seq, pre- and post-anti-PD1 immunotherapy.

**Key question**: Are these tumor-reactive T cells?  
Yost 2019 does **not** include a direct tumor-reactivity assay. Instead, exhaustion state  
(`CD8_ex`, `CD8_ex_act`) and clonal expansion are used as **proxies** for tumor reactivity.

| Column | Meaning |
|--------|---------|
| `cluster` | Original Yost 2019 cell-type annotation (9 types) |
| `clonotype_id` | TCR CDR3 amino acid sequence (NA if no TCR detected) |
| `has_tcr` | Boolean — TCR detected |
| `treatment` | `pre` or `post` anti-PD1 |
| `patient` | 11 patients: su001–su012 (no su011) |


In [ ]:
suppressPackageStartupMessages({
  .libPaths(c("renv/library/macos/R-4.4/aarch64-apple-darwin20", .libPaths()))
  library(Seurat)
  library(ggplot2)
  library(dplyr)
  library(tidyr)
})

options(repr.plot.width = 10, repr.plot.height = 4.5)

# Load all per-patient query objects to get the full dataset
patients <- c("su001","su002","su003","su004","su005",
              "su006","su007","su008","su009","su010","su012")
all_meta <- lapply(patients, function(pt) {
  obj <- readRDS(paste0("data/cca/query_", pt, ".rds"))
  obj@meta.data$cell_id <- colnames(obj)
  obj@meta.data
})
meta <- do.call(rbind, all_meta)
rownames(meta) <- NULL

cat("Total cells:", nrow(meta), "\n")
cat("Patients:", length(unique(meta$patient)), "\n")
cat("Columns:", paste(colnames(meta), collapse=", "), "\n")


## 1. Dataset Overview

In [ ]:
cat("=== Cell-type distribution ===\n")
ct <- sort(table(meta$cluster), decreasing=TRUE)
print(ct)
cat("\n=== Treatment ===\n")
print(table(meta$treatment))
cat("\n=== TCR detection ===\n")
print(table(meta$has_tcr))
cat("  TCR detection rate:", round(100*mean(meta$has_tcr), 1), "%\n")
cat("\n=== Cells per patient ===\n")
print(sort(table(meta$patient), decreasing=TRUE))


## 2. Cell-type Bar Chart

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4)
ct_df <- as.data.frame(ct)
ct_df$pct <- round(100 * ct_df$Freq / sum(ct_df$Freq), 1)

p <- ggplot(ct_df, aes(x = reorder(Var1, Freq), y = Freq, fill = Var1)) +
  geom_col(show.legend = FALSE) +
  coord_flip() +
  geom_text(aes(label = paste0(pct, "%")), hjust = -0.1, size = 3) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
  labs(title = "T-cell subtypes — Yost 2019 BCC (all 11 patients)",
       x = NULL, y = "Cells") +
  theme_classic(base_size = 11)
print(p)


## 3. UMAP — All Cells

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5)

p1 <- ggplot(meta, aes(UMAP1, UMAP2, colour = cluster)) +
  geom_point(size = 0.1, alpha = 0.4) +
  theme_void(base_size = 10) +
  guides(colour = guide_legend(override.aes = list(size=3, alpha=1))) +
  labs(title = "UMAP — Cell Type", colour = NULL)

p2 <- ggplot(meta, aes(UMAP1, UMAP2, colour = treatment)) +
  geom_point(size = 0.1, alpha = 0.4) +
  scale_colour_manual(values = c("pre"="#457b9d","post"="#e63946")) +
  theme_void(base_size = 10) +
  guides(colour = guide_legend(override.aes = list(size=3, alpha=1))) +
  labs(title = "UMAP — Treatment", colour = NULL)

gridExtra::grid.arrange(p1, p2, ncol = 2)


## 4. Are These Tumor-reactive? — Exhaustion as a Proxy

In [ ]:
cat("Yost 2019 does NOT have a direct tumor-reactivity assay.\n")
cat("Tumor-reactive T cells are inferred from:\n")
cat("  1) Cell state: CD8_ex (exhausted) and CD8_ex_act (exhausted+activated)\n")
cat("     are the strongest proxies — exhaustion is driven by chronic antigen\n")
cat("     stimulation from tumor neoantigens.\n")
cat("  2) Clonal expansion: tumor-reactive clones tend to be large.\n")
cat("  3) Pre→post treatment dynamics: anti-PD1 reinvigorates CD8_ex cells.\n\n")

# Exhaustion category
meta$is_exhausted <- meta$cluster %in% c("CD8_ex", "CD8_ex_act")
cat("Exhausted cells (CD8_ex + CD8_ex_act):\n")
print(table(meta$is_exhausted))
cat("Exhaustion rate:", round(100*mean(meta$is_exhausted), 1), "%\n")


In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4.5)

p_ex <- ggplot(meta %>% arrange(is_exhausted),
               aes(UMAP1, UMAP2,
                   colour = is_exhausted,
                   alpha  = is_exhausted)) +
  geom_point(size = 0.15) +
  scale_colour_manual(values = c("FALSE"="grey80", "TRUE"="#e63946"),
                      labels = c("Other T cells", "Exhausted (CD8_ex/ex_act)")) +
  scale_alpha_manual(values = c("FALSE"=0.15, "TRUE"=0.7), guide="none") +
  theme_void(base_size=10) +
  guides(colour=guide_legend(override.aes=list(size=3, alpha=1))) +
  labs(title="UMAP — Exhausted CD8 T cells (putative tumor-reactive)", colour=NULL)
print(p_ex)


## 5. Clonal Expansion

In [ ]:
# Clonotype sizes
clo_size <- meta %>%
  filter(!is.na(clonotype_id)) %>%
  count(clonotype_id, cluster, name="clone_size") %>%
  arrange(desc(clone_size))

cat("=== Top 20 clonotypes by size ===\n")
print(head(clo_size, 20))

cat("\n=== Clonotype size distribution ===\n")
cat("Unique clonotypes:", length(unique(na.omit(meta$clonotype_id))), "\n")
cat("Cells with TCR:", sum(!is.na(meta$clonotype_id)), "\n")


In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4)

# Clone size by cell type (how large are exhausted clones?)
clone_ct <- meta %>%
  filter(!is.na(clonotype_id)) %>%
  group_by(clonotype_id, cluster) %>%
  summarise(n=n(), .groups="drop")

p_box <- ggplot(clone_ct, aes(x = reorder(cluster, n, median),
                               y = n, fill = cluster)) +
  geom_boxplot(show.legend=FALSE, outlier.size=0.5) +
  coord_flip() +
  scale_y_log10() +
  labs(title="Clone size distribution by cell type",
       x=NULL, y="Clone size (log scale)") +
  theme_classic(base_size=11)
print(p_box)


## 6. Pre vs Post Treatment — Exhaustion Shift

In [ ]:
options(repr.plot.width = 11, repr.plot.height = 4.5)

treat_ct <- meta %>%
  group_by(treatment, cluster) %>%
  summarise(n=n(), .groups="drop") %>%
  group_by(treatment) %>%
  mutate(pct=round(100*n/sum(n),1))

p_treat <- ggplot(treat_ct, aes(x=cluster, y=pct, fill=treatment)) +
  geom_col(position="dodge", alpha=0.85) +
  scale_fill_manual(values=c("pre"="#457b9d","post"="#e63946")) +
  labs(title="Cell-type proportions: pre vs post anti-PD1",
       x=NULL, y="% of cells", fill="Treatment") +
  theme_classic(base_size=10) +
  theme(axis.text.x=element_text(angle=35, hjust=1))
print(p_treat)

cat("\n=== Exhausted % by treatment ===\n")
meta %>%
  group_by(treatment) %>%
  summarise(pct_exhausted=round(100*mean(is_exhausted),1)) %>%
  print()


## 7. Per-patient Exhaustion

In [ ]:
options(repr.plot.width = 11, repr.plot.height = 4)

pat_ex <- meta %>%
  group_by(patient, treatment) %>%
  summarise(n_total=n(),
            n_ex=sum(is_exhausted),
            pct_ex=round(100*mean(is_exhausted),1),
            .groups="drop")
print(pat_ex)

p_pat <- ggplot(pat_ex, aes(x=reorder(patient, pct_ex), y=pct_ex, fill=treatment)) +
  geom_col(alpha=0.85) +
  coord_flip() +
  scale_fill_manual(values=c("pre"="#457b9d","post"="#e63946")) +
  labs(title="% Exhausted CD8 cells per patient (pre/post)",
       x="Patient", y="% exhausted", fill="Treatment") +
  theme_classic(base_size=11)
print(p_pat)


## 8. Summary

In [ ]:
cat("╔══════════════════════════════════════════════════════════╗\n")
cat("║  GSE123813 BCC T-cell dataset — key facts               ║\n")
cat("╚══════════════════════════════════════════════════════════╝\n\n")

cat("Dataset: Yost et al. 2019 (BCC, anti-PD1)\n")
cat("  Total T cells :", nrow(meta), "\n")
cat("  Patients       :", length(unique(meta$patient)), "\n")
cat("  Cell types     :", length(unique(meta$cluster)), "\n")
cat("  Treatment      : pre=", sum(meta$treatment=="pre"),
    " post=", sum(meta$treatment=="post"), "\n\n")

cat("TCR repertoire:\n")
cat("  Cells with TCR  :", sum(!is.na(meta$clonotype_id)), "\n")
cat("  Unique clones   :", length(unique(na.omit(meta$clonotype_id))), "\n\n")

cat("Tumor reactivity:\n")
cat("  → No direct reactivity assay — exhaustion state and clonal expansion\n")
cat("    are used as proxies for tumor-reactive CD8 T cells.\n")
cat("  → CD8_ex + CD8_ex_act =", sum(meta$is_exhausted), "cells (",
    round(100*mean(meta$is_exhausted),1), "%) are PROXIES for tumor-reactive CD8\n")
